In [2]:
import sys, json; sys.path.append("..")
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_predict, RandomizedSearchCV
from src.train import make_lgbm
from src.evaluate import report

train = pd.read_csv("../data/processed/train.csv", index_col=0)
y = train.pop("SeriousDlqin2yrs")
X = train
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

proba = cross_val_predict(make_lgbm(), X, y, cv=cv, method="predict_proba")[:, 1]
report(y, (proba >= 0.5).astype(int), proba)

{'accuracy': 0.8076317302644189,
 'precision': 0.22338120248282956,
 'recall': 0.7582595686323401,
 'roc_auc': 0.863028258068636,
 'pr_auc': 0.39747650794220474,
 'TN': np.int64(90833),
 'FP': np.int64(21145),
 'FN': np.int64(1939),
 'TP': np.int64(6082)}

In [3]:
param_dist = {
    "n_estimators": [200, 400, 600],
    "learning_rate": [0.02, 0.05, 0.1],
    "num_leaves": [15, 31, 63],
    "min_child_samples": [20, 50, 100],
    "colsample_bytree": [0.7, 0.9, 1.0],
}
search = RandomizedSearchCV(make_lgbm(), param_dist, n_iter=12, scoring="average_precision",
                            cv=cv, n_jobs=-1, random_state=42)
search.fit(X, y)
print(search.best_params_, round(search.best_score_, 4))

{'num_leaves': 15, 'n_estimators': 600, 'min_child_samples': 100, 'learning_rate': 0.02, 'colsample_bytree': 0.7} 0.4015


In [4]:
best = search.best_params_
proba = cross_val_predict(make_lgbm(**best), X, y, cv=cv, method="predict_proba")[:, 1]
print(report(y, (proba >= 0.5).astype(int), proba))

import os; os.makedirs("../reports", exist_ok=True)
json.dump(best, open("../reports/best_lgbm_params.json", "w"))

{'accuracy': 0.8021900182501521, 'precision': 0.21929699221261698, 'recall': 0.7653659144745044, 'roc_auc': 0.865129192039964, 'pr_auc': 0.4006807552651483, 'TN': np.int64(90123), 'FP': np.int64(21855), 'FN': np.int64(1882), 'TP': np.int64(6139)}
